# Feature Engineering

In [53]:
#importando bibliotecas
import os

import pandas as pd
import joblib
import torch
from sklearn.preprocessing import MinMaxScaler

In [54]:
df = pd.read_pickle('data/raw/raw_data.pkl')
df.head()

,Date,Close,High,Low,Open,Volume
0,2018-01-02,40.267078,40.276431,39.565806,39.776190,102223600
1,2018-01-03,40.260056,40.802375,40.196944,40.330184,118071600
2,2018-01-04,40.447067,40.549921,40.224998,40.332525,89738400
3,2018-01-05,40.907570,40.994059,40.451743,40.542909,94640000
4,2018-01-08,40.755642,41.050175,40.657461,40.755642,82271200


### Feature Engineering

A engenharia de atributos é a etapa de preparação que transforma os dados brutos em um formato que a rede neural consiga processar, garantindo estabilidade matemática e contexto temporal ao modelo.

*   **Normalização (`MinMaxScaler`)**: Redimensiona todos os preços para uma escala entre 0 e 1. Esse processo é vital para redes neurais, pois evita que valores de grande magnitude causem instabilidade no cálculo dos gradientes e acelera a convergência do treinamento.
*   **Criação de janelas temporais**: Transforma a série temporal em um problema de aprendizado supervisionado através de uma "janela deslizante". O código agrupa sequências de 60 dias como entrada (**X**) e define o preço do dia seguinte como o alvo (**y**), fornecendo ao modelo o contexto histórico necessário para identificar tendências.
*   **Separação de dados de treino e teste**: Separa os dados em 80% para treinamento e 20% para teste de forma cronológica. Diferente de outros modelos de ML, em séries temporais não se deve embaralhar os dados (*shuffle*), pois a ordem dos eventos é a informação mais importante para o aprendizado.
*   **Conversão dos dados para tensores (`FloatTensor`)**: Converte as listas de dados do formato NumPy para o formato de Tensores do PyTorch. Além de definir a precisão decimal (float32) exigida pelo modelo, essa conversão prepara os dados para serem processados eficientemente pela arquitetura de Deep Learning do Pytorch.

In [55]:
#definindo técnica de normalização
scaler = MinMaxScaler(feature_range=(0, 1))

data = df['Close'].values.reshape(-1, 1)
data_scaled = scaler.fit_transform(data)

#salva os parâmetros da normalização para permitir a "desnormalização" futura, transformando as previsões da rede de volta para os valores reais em moeda
joblib.dump(scaler, 'models/scaler.pkl') 

['models/scaler.pkl']

In [56]:
#criação de janleas temporais
X, y = [], []
for i in range(len(data_scaled) - 60):
    X.append(data_scaled[i : i+60])
    y.append(data_scaled[i+60])

#separação de dados de treino e teste e conversão dos dados para tensores
split = int(len(X) * 0.8)
data = {
    'X_train': torch.FloatTensor(X[:split]),
    'y_train': torch.FloatTensor(y[:split]),
    'X_test':  torch.FloatTensor(X[split:]),
    'y_test':  torch.FloatTensor(y[split:])
}
data

{'X_train': tensor([[[0.0231],
          [0.0231],
          [0.0238],
          ...,
          [0.0241],
          [0.0204],
          [0.0189]],
 
         [[0.0231],
          [0.0238],
          [0.0254],
          ...,
          [0.0204],
          [0.0189],
          [0.0200]],
 
         [[0.0238],
          [0.0254],
          [0.0248],
          ...,
          [0.0189],
          [0.0200],
          [0.0191]],
 
         ...,
 
         [[0.6395],
          [0.6449],
          [0.6580],
          ...,
          [0.6891],
          [0.6743],
          [0.6638]],
 
         [[0.6449],
          [0.6580],
          [0.6596],
          ...,
          [0.6743],
          [0.6638],
          [0.6606]],
 
         [[0.6580],
          [0.6596],
          [0.6702],
          ...,
          [0.6638],
          [0.6606],
          [0.6657]]]),
 'y_train': tensor([[0.0200],
         [0.0191],
         [0.0205],
         ...,
         [0.6606],
         [0.6657],
         [0.6631]]),
 'X_

In [57]:
torch.save(data, 'data/processed/training_data.pt')